In [15]:
from openai import OpenAI
import json

client = OpenAI()
messages = []


In [17]:
import os
from dotenv import load_dotenv
import requests
import json

load_dotenv()
MOVIE_BASE_API = os.getenv("MOVIE_BASE_API")

def call_movie_api(endpoint: str):
    url = f"{MOVIE_BASE_API}/{endpoint}"

    try:
        response = requests.get(url)
        return response.json()
    except Exception as e:
        print(f"[Error] API 호출 실패: {e}")
        return None
        
def get_popular_movies():
    data = call_movie_api("/movies")
    return json.dumps(data, ensure_ascii=False)

def get_movie_details(id: int):
    data = call_movie_api(f"/movies/{id}")
    return json.dumps(data, ensure_ascii=False)

def get_movie_credits(id: int):
    data = call_movie_api(f"/movies/{id}/credits")
    return json.dumps(data, ensure_ascii=False)

def get_similar_movies(id: int):
    data = call_movie_api(f"/movies/{id}/similar")
    return json.dumps(data, ensure_ascii=False)

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
}

In [18]:
from openai.types.chat import ChatCompletionMessage


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID를 기반으로 해당 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "조회할 영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID를 기반으로 해당 영화의 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "출연진을 조회할 영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "영화 ID를 기준으로 비슷한 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "기준이 되는 영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

def process_ai_response(message: ChatCompletionMessage):

    # tool_call이 존재할 경우 message에 툴콜 정보를 저장.
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # tool_call이 존재할 경우 해당 툴콜을 실행.
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            # 문자열인 형태인 JSON을 딕셔너리로 변환. ex) "{"city":"seoul"}" -> {"city": "seoul"}
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            # **를 사용하여 딕셔너리를 풀어서 함수에 전달. ex) {"city": "seoul"} -> city="seoul"
            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")
            
            # 함수 실행 결과를 메모리에 저장.
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        # AI가 tool_call 실행으로 인해 추가된 새로운 messages를 대화에서 볼 수 있게 함.
        call_ai()
    # tool_call이 존재하지 않을 경우 일반적인 대화 흐름.
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

# OpenAI API를 통해 대화를 진행
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)    

In [ ]:
# 반복문을 통해 Agent와 대화를 진행
while True:
    message = input("Send a message to LLM...")
    if message == "quit":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()